In [0]:
import pandas as pd
from pyspark.sql import functions as F

# Load cleaned silver table
df_silver = spark.table("`data-1`.`example-1`.`cleaned_silver`")
 


In [0]:
# Extract day of week from signup_date and count signups per day
df_days = df_silver.withColumn('day_of_week', F.dayofweek('signup_date')) \
    .withColumn('day_name', 
        F.when(F.col('day_of_week') == 1, 'Sunday')
        .when(F.col('day_of_week') == 2, 'Monday')
        .when(F.col('day_of_week') == 3, 'Tuesday')
        .when(F.col('day_of_week') == 4, 'Wednesday')
        .when(F.col('day_of_week') == 5, 'Thursday')
        .when(F.col('day_of_week') == 6, 'Friday')
        .otherwise('Saturday')
    ) \
    .groupBy('day_name', 'day_of_week') \
    .agg(F.count('*').alias('signup_count')) \
    .orderBy(F.desc('signup_count'))

display(df_days)

In [0]:
# Count signups by referral source (ads)
df_referrals = df_silver.groupBy('referral_source') \
    .agg(F.count('*').alias('click_count')) \
    .orderBy(F.desc('click_count'))

display(df_referrals)

In [0]:
# Create insights combining best days and top referral sources
insights_gold = df_days.select(
    F.lit('Best Signup Days').alias('insight_type'),
    F.col('day_name').alias('dimension'),
    F.col('signup_count').alias('metric_value')
).union(
    df_referrals.select(
        F.lit('Top Referral Sources').alias('insight_type'),
        F.col('referral_source').alias('dimension'),
        F.col('click_count').alias('metric_value')
    )
)


# Save to gold table
insights_gold.write.mode("overwrite").saveAsTable("`data-1`.`example-1`.`insights_gold`")


In [0]:
display(insights_gold)